# KhushiDL - Heart Disease Detector

In [225]:
# Importing necessary modules
import tensorflow as tf
import pandas as pd

## Data Preprocessing

In [226]:
# Loading the data
train_data = pd.read_csv('data/train.csv', header=None)
validation_data = pd.read_csv('data/validation.csv', header=None)
test_data = pd.read_csv('data/test.csv', header=None)

In [227]:
print("Shape: ", train_data.shape)
train_data.info()

Shape:  (644, 14)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 644 entries, 0 to 643
Data columns (total 14 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   0       644 non-null    float64
 1   1       644 non-null    float64
 2   2       644 non-null    float64
 3   3       644 non-null    float64
 4   4       644 non-null    float64
 5   5       644 non-null    float64
 6   6       644 non-null    float64
 7   7       644 non-null    float64
 8   8       644 non-null    float64
 9   9       644 non-null    float64
 10  10      644 non-null    float64
 11  11      644 non-null    float64
 12  12      644 non-null    float64
 13  13      644 non-null    float64
dtypes: float64(14)
memory usage: 70.6 KB


In [228]:
# seperating params
train_x = train_data.iloc[:, [x for x in range(13)]]
validation_x = validation_data.iloc[:, [x for x in range(13)]]
test_x = test_data.iloc[:, [x for x in range(13)]]

In [229]:
print("Train Shape: ", train_x.shape)
print("Validation Shape: ", validation_x.shape)
print("Test Shape: ", test_x.shape)

Train Shape:  (644, 13)
Validation Shape:  (138, 13)
Test Shape:  (138, 13)


In [230]:
# sperating labels
train_y = train_data.iloc[: , [13]].copy()
validation_y = validation_data.iloc[: , [13]].copy()
test_y = test_data.iloc[: , [13]].copy()

In [231]:
print("Train Shape: ", train_y.shape)
print("Validation Shape: ", validation_y.shape)
print("Test Shape: ", test_y.shape)

Train Shape:  (644, 1)
Validation Shape:  (138, 1)
Test Shape:  (138, 1)


In [232]:
test_y.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 138 entries, 0 to 137
Data columns (total 1 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   13      138 non-null    float64
dtypes: float64(1)
memory usage: 1.2 KB


### Data Pipeline

In [233]:
BATCH = 64
AUTOTUNE = tf.data.AUTOTUNE


train_ds = (
    tf.data.Dataset.from_tensor_slices((train_x, train_y))
    .batch(BATCH)
    .shuffle(buffer_size=train_x.shape[0])
    .prefetch(AUTOTUNE)
)

validation_ds = (
    tf.data.Dataset.from_tensor_slices((validation_x, validation_y))
    .batch(BATCH)
    .prefetch(AUTOTUNE)
)

test_ds = (
    tf.data.Dataset.from_tensor_slices((test_x, test_y))
    .batch(BATCH)
    .prefetch(AUTOTUNE)
)

## Model Architecture

In [234]:
# creating the actual model
model = tf.keras.Sequential([
    tf.keras.Input(shape=(13,)),
    
    tf.keras.layers.Dense(100, activation='relu'),
    tf.keras.layers.Dense(50, activation='relu'),
    
    tf.keras.layers.Dense(10, activation='softmax')
])

model.summary()

Model: "sequential_14"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_46 (Dense)                │ (None, 100)            │         1,400 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_47 (Dense)                │ (None, 50)             │         5,050 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_48 (Dense)                │ (None, 10)             │           510 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 6,960 (27.19 KB)

 Trainable params: 6,960 (27.19 KB)

 Non-trainable params: 0 (0.00 B)

### Compiling and Training the Model

In [235]:
# Compiling the model
model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

In [236]:
# EarlyStopping callback
earlyStopper = tf.keras.callbacks.EarlyStopping(
    monitor='val_loss',
    patience=5,
    restore_best_weights=True
)

In [237]:
# training the model
model_history = model.fit(
    train_ds,
    validation_data = validation_ds,
    epochs=50,
    callbacks=[earlyStopper]
)

Epoch 1/50
11/11 ━━━━━━━━━━━━━━━━━━━━ 1s 11ms/step - accuracy: 0.2252 - loss: 17.1085 - val_accuracy: 0.4348 - val_loss: 7.8139
Epoch 2/50
11/11 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.3665 - loss: 5.8376 - val_accuracy: 0.2536 - val_loss: 5.6992
Epoch 3/50
11/11 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.4037 - loss: 3.4950 - val_accuracy: 0.3768 - val_loss: 2.9045
Epoch 4/50
11/11 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.4224 - loss: 2.1197 - val_accuracy: 0.3841 - val_loss: 2.3009
Epoch 5/50
11/11 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.4348 - loss: 1.9654 - val_accuracy: 0.3841 - val_loss: 2.0599
Epoch 6/50
11/11 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.4301 - loss: 1.8540 - val_accuracy: 0.4420 - val_loss: 1.8198
Epoch 7/50
11/11 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.4146 - loss: 1.7852 - val_accuracy: 0.3913 - val_loss: 1.7247
Epoch 8/50
11/11 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.4425 - loss: 1.7165 - val_accuracy: 0.4348 - val_los

In [238]:
model.evaluate(test_ds)

3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.4638 - loss: 1.3661


[1.3661341667175293, 0.4637681245803833]